# Notebook 02 — EDA y Pipeline de Limpieza
**Proyecto:** Sistema de Clasificación de Riesgo Académico Estudiantil  
**Dataset:** Students Performance Dataset (Kaggle, 2024)  
**Fecha:** Junio 16 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Librerías cargadas correctamente.')

## 1. Carga y diagnóstico inicial

In [ ]:
df = pd.read_csv('../data/raw/students_performance.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('--- Tipos de datos ---')
print(df.dtypes)
print('\n--- Valores nulos por columna ---')
print(df.isnull().sum())
print('\n--- Registros duplicados ---')
print(df.duplicated().sum())

In [ ]:
print('--- Estadísticas descriptivas ---')
df.describe()

**Diagnóstico:** El dataset presenta calidad excepcional: 0 valores nulos y 0 duplicados. No se requiere imputación ni eliminación de registros.

## 2. Análisis de la variable objetivo

In [ ]:
label_map = {0: 'A (Exc)', 1: 'B (Bue)', 2: 'C (Pro)', 3: 'D (Baj)', 4: 'F (Rep)'}
dist = df['GradeClass'].value_counts().sort_index()
print('Distribución GradeClass:')
for k, v in dist.items():
    print(f'  Clase {k} [{label_map[k]}]: {v} ({v/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c']
ax.bar([label_map[i] for i in dist.index], dist.values, color=colors)
ax.set_title('Distribución de la Variable Objetivo (GradeClass)')
ax.set_xlabel('Categoría')
ax.set_ylabel('Frecuencia')
for i, v in enumerate(dist.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/distribucion_target_02.png', dpi=120)
plt.show()
print('Desbalance moderado -> justifica F1-score macro como métrica principal.')

## 3. Análisis de data leakage — variable GPA

In [ ]:
corr_gpa = df[['GPA', 'GradeClass']].corr().iloc[0, 1]
print(f'Correlación GPA ↔ GradeClass: {corr_gpa:.4f}')
print()
print('CONCLUSIÓN: GPA tiene correlación -0.97 con GradeClass.')
print('El GPA es esencialmente el origen del que se derivó GradeClass.')
print('Incluirlo en el modelo constituye DATA LEAKAGE grave.')
print('→ GPA EXCLUIDO del conjunto de features.')

## 4. EDA de variables de entrada

In [ ]:
# Correlaciones entre features (sin GPA ni StudentID)
features = df.drop(columns=['StudentID', 'GPA', 'GradeClass'])
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = features.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de Correlación — Variables de Entrada')
plt.tight_layout()
plt.savefig('../docs/correlacion_features.png', dpi=120)
plt.show()

In [ ]:
# Distribución de StudyTimeWeekly y Absences por clase
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls in sorted(df['GradeClass'].unique()):
    subset = df[df['GradeClass'] == cls]
    axes[0].hist(subset['StudyTimeWeekly'], alpha=0.5, label=label_map[cls], bins=15)
    axes[1].hist(subset['Absences'], alpha=0.5, label=label_map[cls], bins=15)
axes[0].set_title('StudyTimeWeekly por Clase')
axes[0].set_xlabel('Horas/semana')
axes[0].legend()
axes[1].set_title('Absences por Clase')
axes[1].set_xlabel('Ausencias')
axes[1].legend()
plt.tight_layout()
plt.savefig('../docs/dist_study_absences.png', dpi=120)
plt.show()

## 5. Pipeline de preparación y guardado

In [ ]:
# Paso 1: Eliminar columnas no predictivas y con data leakage
df_clean = df.drop(columns=['StudentID', 'GPA'])
print('Columnas eliminadas: StudentID (identificador), GPA (data leakage)')
print('Shape después de limpieza:', df_clean.shape)

# Paso 2: No hay nulos ni duplicados -> confirmado arriba

# Paso 3: Verificar tipos finales
print('\nTipos finales:')
print(df_clean.dtypes)

In [ ]:
# Paso 4: Separar X e y
X = df_clean.drop(columns=['GradeClass'])
y = df_clean['GradeClass']

# Paso 5: Split train/test estratificado, semilla fija
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas')
print('Proporción train:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Proporción test: ', y_test.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Paso 6: Escalado de variables continuas (fit en train, transform en test)
cont_cols = ['StudyTimeWeekly', 'Absences', 'Age']
scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc = X_test.copy()
X_train_sc[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test_sc[cont_cols] = scaler.transform(X_test[cont_cols])  # <-- solo transform, NO fit
print('Escalado aplicado correctamente (sin fuga de datos).')
print('Media train StudyTimeWeekly (tras escalar):', X_train_sc['StudyTimeWeekly'].mean().round(6))

In [ ]:
# Paso 7: Guardar dataset procesado
Path('../data/processed').mkdir(parents=True, exist_ok=True)
df_clean.to_csv('../data/processed/dataset_limpio.csv', index=False)
print('Dataset limpio guardado en data/processed/dataset_limpio.csv')
print('Shape final:', df_clean.shape)

## Resumen del pipeline

| Paso | Acción | Justificación |
|------|--------|---------------|
| 1 | Eliminación de `StudentID` | Identificador sin valor predictivo |
| 2 | Eliminación de `GPA` | Correlación -0.97 con target → data leakage |
| 3 | Sin tratamiento de nulos | Dataset sin valores faltantes |
| 4 | Sin eliminación de duplicados | Sin registros duplicados |
| 5 | Split 80/20 estratificado, seed=42 | Reproducibilidad y proporción de clases |
| 6 | StandardScaler solo en continuas | Fit en train, transform en test |
| 7 | Guardado en `data/processed/` | Separación raw vs procesado |